# Learning Material CRUD Operations

This notebook covers working with Learning Materials and Learning Material Types.

## Learning Material Type Model

- **id** (Integer, Primary Key) - Auto-generated
- **name** (String, max 20 chars) - Required, unique

## Learning Material Model

- **id** (Integer, Primary Key) - Auto-generated
- **name** (String, max 100 chars) - Required
- **description** (Text) - Optional
- **link** (String, max 100 chars) - Optional
- **price** (Integer) - Optional
- **type_id** (Integer, Foreign Key) - Required
- **tag_ids** (Integer Array) - Optional, PostgreSQL array
- **wiki_note_ids** (Integer Array) - Optional, PostgreSQL array
- **notes** (JSONB) - Optional, stored as text

In [ ]:
# Setup imports
import sys
sys.path.insert(0, '..')

from teamup.database import get_session
from teamup.repository import LearningMaterialTypeRepository, LearningMaterialRepository
import json

print("✓ Imports ready")

## Part 1: Learning Material Types

First, let's work with material types (Book, Video, Course, etc.).

In [ ]:
# Create material types
with get_session() as session:
    repo = LearningMaterialTypeRepository(session)
    
    # Check if types already exist
    existing = repo.get_all()
    if len(existing) == 0:
        print("Creating material types...")
        book_type = repo.create(name="Book")
        video_type = repo.create(name="Video")
        course_type = repo.create(name="Online Course")
        article_type = repo.create(name="Article")
        
        print(f"Created: {book_type}")
        print(f"Created: {video_type}")
        print(f"Created: {course_type}")
        print(f"Created: {article_type}")
    else:
        print(f"Found {len(existing)} existing types:")
        for t in existing:
            print(f"  - {t}")

In [ ]:
# Retrieve types
with get_session() as session:
    repo = LearningMaterialTypeRepository(session)
    
    # Get all types
    all_types = repo.get_all()
    print(f"\nAll material types ({len(all_types)}):")
    for t in all_types:
        print(f"  ID {t.id}: {t.name}")

In [ ]:
# Get type by name
with get_session() as session:
    repo = LearningMaterialTypeRepository(session)
    
    book_type = repo.get_by_name("Book")
    if book_type:
        print(f"Found type: {book_type}")
        print(f"Type ID: {book_type.id}")

## Part 2: Learning Materials

Now let's create and manage learning materials.

In [ ]:
# Get a type ID for creating materials
with get_session() as session:
    type_repo = LearningMaterialTypeRepository(session)
    book_type = type_repo.get_by_name("Book")
    course_type = type_repo.get_by_name("Online Course")
    
    book_type_id = book_type.id if book_type else None
    course_type_id = course_type.id if course_type else None
    
    print(f"Book type ID: {book_type_id}")
    print(f"Course type ID: {course_type_id}")

In [ ]:
# Create learning materials with all features
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    # Book with all fields
    python_book = repo.create(
        name="Python for Data Science",
        description="Comprehensive guide to Python programming for data analysis and machine learning",
        link="https://example.com/python-book",
        price=4500,  # $45.00
        type_id=book_type_id,
        tag_ids=[],  # Empty array (avoid actual tag IDs due to DB triggers)
        wiki_note_ids=[],  # Empty array
        notes=json.dumps({"difficulty": "intermediate", "pages": 450})
    )
    print(f"Created: {python_book}")
    
    # Course with minimal fields
    ml_course = repo.create(
        name="Machine Learning Fundamentals",
        type_id=course_type_id
    )
    print(f"Created: {ml_course}")

In [ ]:
# Retrieve all materials
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    all_materials = repo.get_all()
    print(f"\nTotal learning materials: {len(all_materials)}")
    for material in all_materials[:5]:  # Show first 5
        print(f"  - {material}")

## Search and Filter

Find materials by various criteria.

In [ ]:
# Search by name
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    results = repo.find_by_name("Python")
    print(f"\nMaterials with 'Python' in name: {len(results)}")
    for material in results:
        print(f"  - {material.name}")

In [ ]:
# Find by type
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    books = repo.find_by_type(book_type_id)
    print(f"\nBooks: {len(books)}")
    for book in books[:5]:
        print(f"  - {book.name}")

In [ ]:
# Find by price range
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    affordable = repo.find_by_price_range(min_price=0, max_price=5000)  # Up to $50
    print(f"\nMaterials under $50: {len(affordable)}")
    for material in affordable[:5]:
        price_str = f"${material.price/100:.2f}" if material.price else "Free"
        print(f"  - {material.name}: {price_str}")

## Update Materials

Modify existing learning materials.

In [ ]:
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    # Find Python book
    materials = repo.find_by_name("Python for Data Science")
    if materials:
        python_book = materials[0]
        print(f"Before: {python_book}")
        
        # Update with new information
        updated = repo.update(
            python_book.id,
            description="Updated: Master Python for data science and ML",
            price=3999,  # Sale price $39.99
            notes=json.dumps({"difficulty": "intermediate", "pages": 450, "edition": 2})
        )
        print(f"After: {updated}")

## Working with Relationships

Materials have a foreign key relationship with types.

In [ ]:
# Show materials with their types
with get_session() as session:
    material_repo = LearningMaterialRepository(session)
    type_repo = LearningMaterialTypeRepository(session)
    
    materials = material_repo.get_all()[:5]
    print("\nMaterials with their types:")
    for material in materials:
        mat_type = type_repo.get_by_id(material.type_id)
        type_name = mat_type.name if mat_type else "Unknown"
        print(f"  [{type_name}] {material.name}")

## Working with PostgreSQL Arrays

Materials support PostgreSQL INTEGER[] arrays for tag_ids and wiki_note_ids.

In [ ]:
# Create material with empty arrays (safest approach)
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    material = repo.create(
        name="Introduction to PostgreSQL",
        type_id=book_type_id,
        tag_ids=[],  # Empty array
        wiki_note_ids=[]  # Empty array
    )
    print(f"Created material with arrays: {material}")
    print(f"  tag_ids: {material.tag_ids}")
    print(f"  wiki_note_ids: {material.wiki_note_ids}")

## Working with JSONB Notes

The notes field stores JSON data as text.

In [ ]:
# Create material with JSON notes
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    notes_data = {
        "difficulty": "beginner",
        "duration_hours": 10,
        "prerequisites": ["Basic programming"],
        "topics": ["Databases", "SQL", "Indexing"]
    }
    
    material = repo.create(
        name="Database Design Course",
        type_id=course_type_id,
        notes=json.dumps(notes_data)
    )
    print(f"\nCreated: {material}")
    print(f"Notes (raw): {material.notes}")
    
    # Parse JSON notes
    if material.notes:
        parsed_notes = json.loads(material.notes)
        print(f"\nParsed notes:")
        print(f"  Difficulty: {parsed_notes['difficulty']}")
        print(f"  Duration: {parsed_notes['duration_hours']} hours")
        print(f"  Topics: {', '.join(parsed_notes['topics'])}")

## Delete Operations

Remove materials and types.

In [ ]:
# Delete a material
with get_session() as session:
    repo = LearningMaterialRepository(session)
    
    materials = repo.find_by_name("Database Design Course")
    if materials:
        material = materials[0]
        print(f"Deleting: {material.name}")
        success = repo.delete(material.id)
        print(f"Deleted: {success}")

In [ ]:
# Check counts
with get_session() as session:
    type_repo = LearningMaterialTypeRepository(session)
    material_repo = LearningMaterialRepository(session)
    
    print(f"\nTotal types: {type_repo.count()}")
    print(f"Total materials: {material_repo.count()}")

## Next Steps

Continue with:
- **04_advanced_examples.ipynb** - Complex workflows, batch operations, and advanced queries